In [ ]:
import json
import os
import numpy as np
import librosa
import soundfile as sf
import collections
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm

INPUT_AUDIO_DIR = '/kaggle/input/nsynth-wav/nsynth-train-all/audio'
JSON_PATH = '/kaggle/input/nsynth-wav/nsynth-train-all/examples-train-original.json'
OUTPUT_DIR = '/kaggle/working/processed_nsynth'

TARGET_FAMILIES = ['bass', 'flute', 'brass', 'string'] 

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

for sub in ['train', 'test_single', 'test_overlap']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)

with open(JSON_PATH, 'r') as f:
    metadata = json.load(f)

family_groups = {fam: [] for fam in TARGET_FAMILIES}
for fid, info in metadata.items():
    fam = info['instrument_family_str']
    if fam in TARGET_FAMILIES:
        family_groups[fam].append(fid)

min_size = min(len(family_groups[fam]) for fam in TARGET_FAMILIES)
print(f"Số lượng mẫu tối đa mỗi lớp để cân bằng: {min_size}")

balanced_ids = []
for fam in TARGET_FAMILIES:
    picked = np.random.choice(family_groups[fam], min_size, replace=False)
    balanced_ids.extend(picked)

train_ids, test_ids = train_test_split(balanced_ids, test_size=0.2, random_state=42)

train_pool = collections.defaultdict(list)
for fid in train_ids:
    fam = metadata[fid]['instrument_family_str']
    train_pool[fam].append(fid)

test_pool = collections.defaultdict(list)
for fid in test_ids:
    fam = metadata[fid]['instrument_family_str']
    test_pool[fam].append(fid)

#  Hàm Xử lý Đơn lẻ (Trim 30dB, 16kHz) 
def process_and_save_single(file_list, target_subdir):
    for fid in tqdm(file_list, desc=f"Saving {target_subdir}"):
        in_p = os.path.join(INPUT_AUDIO_DIR, f"{fid}.wav")
        out_p = os.path.join(OUTPUT_DIR, target_subdir, f"{fid}.wav")
        try:
            # Load chuẩn 16kHz của NSynth [cite: 32]
            y, sr = librosa.load(in_p, sr=16000)
            # Loại bỏ silent (giữ lại Transient/Percussive) [cite: 79, 80]
            y_trim, _ = librosa.effects.trim(y, top_db=30)
            sf.write(out_p, y_trim, sr)
        except Exception:
            continue

#  Hàm Tạo Overlap
def create_diverse_overlaps(groups_dict, num_samples=100):
    families = list(groups_dict.keys())
    overlap_log = []
    for i in tqdm(range(num_samples), desc="Creating Overlaps"):
        n_mix = np.random.choice([2, 3])
        picked_fams = np.random.choice(families, n_mix, replace=False)
        
        mixed_y = None
        component_ids = []
        for fam in picked_fams:
            fid = np.random.choice(groups_dict[fam])
            component_ids.append(fid)
            y, _ = librosa.load(os.path.join(INPUT_AUDIO_DIR, f"{fid}.wav"), sr=16000)
            y_trim, _ = librosa.effects.trim(y, top_db=30)
            # Cố định 4s (64000 mẫu) để cộng sóng âm [cite: 31, 33]
            y_fixed = librosa.util.fix_length(y_trim, size=64000)
            if mixed_y is None: mixed_y = y_fixed
            else: mixed_y += y_fixed
        
        # Chuẩn hóa tránh Distortion [cite: 63, 64]
        mixed_y /= np.max(np.abs(mixed_y))
        fname = f"overlap_{i:03d}_{'_'.join(picked_fams)}.wav"
        sf.write(os.path.join(OUTPUT_DIR, 'test_overlap', fname), mixed_y, 16000)
        overlap_log.append({"file": fname, "components": component_ids})
    
    with open(os.path.join(OUTPUT_DIR, 'overlap_log.json'), 'w') as f:
        json.dump(overlap_log, f, indent=4)

final_train_list = []
for fam in TARGET_FAMILIES:
    picked = np.random.choice(train_pool[fam], 300, replace=False)
    final_train_list.extend(picked)

final_test_single_list = []
for fam in TARGET_FAMILIES:
    picked = np.random.choice(test_pool[fam], 50, replace=False)
    final_test_single_list.extend(picked)

process_and_save_single(final_train_list, 'train')
process_and_save_single(final_test_single_list, 'test_single')
create_diverse_overlaps(test_pool, num_samples=100)

print(f" Mỗi lớp có đúng 300 file Train. Tổng: {len(final_train_list)}")

Đã xóa dữ liệu cũ để đảm bảo cân bằng chính xác.
Số lượng mẫu tối đa mỗi lớp để cân bằng: 8773
--- Thực thi lưu file ---


Creating Overlaps: 100%|██████████| 100/100 [00:03<00:00, 28.37it/s]

Hoàn thành! Mỗi lớp có đúng 300 file Train. Tổng: 1200


In [ ]:
import os
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

def check_balance(directory):
    families = []
    for filename in os.listdir(directory):
        if filename.endswith('.wav'):
            family = filename.split('_')[0]
            families.append(family)
    return Counter(families)

train_report = check_balance('/kaggle/working/processed_nsynth/train')
test_single_report = check_balance('/kaggle/working/processed_nsynth/test_single')

df_single = pd.DataFrame({
    'Lớp': TARGET_FAMILIES,
    'Train': [train_report.get(f, 0) for f in TARGET_FAMILIES],
    'Test Single': [test_single_report.get(f, 0) for f in TARGET_FAMILIES]
})

print("--- Báo cáo Dữ liệu Đơn lẻ ---")
print(df_single)

--- Báo cáo Dữ liệu Đơn lẻ ---
      Lớp  Train  Test Single
0    bass    300           50
1   flute    300           50
2   brass    300           50
3  string    300           50


In [ ]:
def check_overlap_balance(directory):
    overlap_counts = Counter()
    for filename in os.listdir(directory):
        if filename.startswith('overlap'):
            parts = filename.replace('.wav', '').split('_')[2:]
            for p in parts:
                overlap_counts[p] += 1
    return overlap_counts

overlap_report = check_overlap_balance('/kaggle/working/processed_nsynth/test_overlap')

df_overlap = pd.DataFrame({
    'Lớp': TARGET_FAMILIES,
    'Tần suất xuất hiện': [overlap_report.get(f, 0) for f in TARGET_FAMILIES]
})

print("\n--- Báo cáo Dữ liệu Chồng chéo (Overlap) ---")
print(df_overlap)


--- Báo cáo Dữ liệu Chồng chéo (Overlap) ---
      Lớp  Tần suất xuất hiện
0    bass                  62
1   flute                  61
2   brass                  71
3  string                  66
